# 08 — Métricas RAGAS (juez vía Ollama + cálculo determinista)

Dos capas: (1) las métricas **deterministas** estilo RAGAS (`rag_baseline/ragas/compute_metrics.py`: context precision/recall, faithfulness por chequeo de claims, relevancy por coseno) y (2) el run **real con la librería `ragas` 0.4.4**, ahora con el juez vía **ChatOllama** (`JUDGE_OLLAMA_MODEL`, por defecto `qwen2.5:7b` — el juez 1.5B en CPU se atoró incluso con timeout de 1800 s por job; en la T4 vía Ollama debería completar). Si el run real no completa, las deterministas siguen siendo las cifras citables, y el notebook lo dice.

**Prerrequisitos:** notebooks 05 y 06 (trazas y retrievals de las tres variantes RAG). El juez no evalúa al grafo ni a MEMO: solo a las tres variantes RAG, como en local.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
JUDGE_OLLAMA_MODEL = "qwen2.5:7b"   # juez: 7b (default, mejor juez) o qwen2.5:1.5b (paridad CPU)
TIME_BUDGET_MIN = 90   # tope de tiempo para el run real con ragas (muestras no hechas = skipped, no numeros)

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 langchain-ollama langchain-huggingface==1.3.0 ragas==0.4.4

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", JUDGE_OLLAMA_MODEL)

## 1. Métricas deterministas (siempre disponibles)

Corre `compute_metrics.py` y `compute_relevancy.py` tal cual: precision y recall de contexto desde los traces de retrieval + ground truth auditable, faithfulness por chequeo de claims contra el contexto condensado, relevancy como proxy de coseno. En local: denso recall 0.47 / faithfulness 0.97, híbrido 0.53 / 0.97, expansión 1.00 / 0.98.

In [ ]:
import subprocess, sys
for script in ["compute_metrics.py", "compute_relevancy.py"]:
    r = subprocess.run([sys.executable, script], cwd=REPO + "/rag_baseline/ragas")
    print(script, "exit:", r.returncode)

## 2. Piloto: una muestra con el juez Ollama

Antes del run completo, una sola muestra (denso Q1, la interesante: el lector contestó 40,236 con 781,585 en contexto) con las 4 métricas y `RunConfig`. Si el piloto no completa en tiempo razonable, el run completo no es viable y nos quedamos con las deterministas — igual que pasó en CPU.

In [ ]:
import json, time
from pathlib import Path
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
try:
    from ragas import RunConfig
except ImportError:
    from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"

judge = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                   temperature=0, num_predict=512)
llm = LangchainLLMWrapper(judge)
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cuda"}))
print("juez Ollama listo:", JUDGE_OLLAMA_MODEL)

trace = [json.loads(l) for l in open(BASE / "run_trace.jsonl")][0]  # denso Q1
gt = json.load(open(RAGAS_DIR / "ground_truth_evidence.json"))["questions"][0]
contexts = [c for c in trace["context"].split("\n\n") if c.strip()]
sample = SingleTurnSample(
    user_input=trace["question"],
    retrieved_contexts=contexts,
    response=trace["final_answer"],
    reference=gt["question"] + " Esperado: Ps. 781,585 millones, +11.2%.",
)
t1 = time.time()
result = evaluate(
    EvaluationDataset(samples=[sample]),
    metrics=[ContextPrecision(), ContextRecall(), Faithfulness(), AnswerRelevancy()],
    llm=llm, embeddings=emb, run_config=RunConfig(timeout=600),
)
print(f"piloto: {time.time() - t1:.1f}s")
print(result.to_pandas().to_string())

## 3. Run real completo (acotado por tiempo)

36 muestras (12 preguntas × 3 variantes), una llamada `evaluate()` por muestra con las 4 métricas, como en el piloto de CPU (`pilot_timeout_900.py`). Tope `TIME_BUDGET_MIN`: las muestras que no se alcanzan quedan como `skipped`, no como números. Al final se comparan los valores juzgados por el LLM con los deterministas, pregunta por pregunta.

In [ ]:
import json, time
from pathlib import Path
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
try:
    from ragas import RunConfig
except ImportError:
    from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"
OUT = RAGAS_DIR / "metrics_ragas_llm_colab.json"

judge = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                   temperature=0, num_predict=512)
llm = LangchainLLMWrapper(judge)
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cuda"}))
METRICS = [ContextPrecision(), ContextRecall(), Faithfulness(), AnswerRelevancy()]
METRIC_NAMES = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]
run_config = RunConfig(timeout=600)

eval_q = {q["id"]: q for q in json.load(open(Path(REPO) / "memo/eval_questions.json"))["questions"]}
det = json.load(open(RAGAS_DIR / "metrics_ragas.json"))["variants"]
gt = {q["id"]: q for q in json.load(open(RAGAS_DIR / "ground_truth_evidence.json"))["questions"]}

VARIANTS = [("dense", "run_trace.jsonl"),
            ("hybrid", "run_trace_hybrid.jsonl"),
            ("expanded", "run_trace_expanded.jsonl")]
traces = {}
for name, fname in VARIANTS:
    traces[name] = {r["id"]: r for r in (json.loads(l) for l in open(BASE / fname))}

results, t_start = {}, time.time()
budget_s = TIME_BUDGET_MIN * 60
for name, _ in VARIANTS:
    for qid in sorted(eval_q):
        if time.time() - t_start > budget_s:
            results[f"{name}/{qid}"] = {"status": "skipped",
                                        "reason": f"tope de tiempo ({TIME_BUDGET_MIN} min)"}
            continue
        tr = traces[name][qid]
        contexts = [c for c in tr["context"].split("\n\n") if c.strip()]
        sample = SingleTurnSample(
            user_input=tr["question"], retrieved_contexts=contexts,
            response=tr["final_answer"],
            reference=gt[qid]["question"] + " Esperado: " + eval_q[qid]["expected"])
        t1 = time.time()
        try:
            res = evaluate(EvaluationDataset(samples=[sample]), metrics=METRICS,
                           llm=llm, embeddings=emb, run_config=run_config)
            row = res.to_pandas().iloc[0].to_dict()
            vals = {m: (None if row.get(m) is None else float(row[m]))
                    for m in METRIC_NAMES}
            results[f"{name}/{qid}"] = {"status": "ok",
                                        "seconds": round(time.time() - t1, 1),
                                        "llm_judged": vals,
                                        "deterministic": det[name]["per_question"][qid]}
        except Exception as e:
            results[f"{name}/{qid}"] = {"status": "error",
                                        "seconds": round(time.time() - t1, 1),
                                        "error": f"{type(e).__name__}: {e}"}
        print(f"{name}/{qid}: {results[f'{name}/{qid}']['status']} "
              f"({results[f'{name}/{qid}'].get('seconds', '?')}s)", flush=True)

done = sum(1 for v in results.values() if v["status"] == "ok")
print(f"\ncompletadas: {done}/36 en {(time.time() - t_start) / 60:.1f} min")
json.dump({"judge": f"ollama {JUDGE_OLLAMA_MODEL} (Colab T4)",
           "results": results}, open(OUT, "w"), ensure_ascii=False, indent=2)
print(f"-> {OUT}")

## 4. Comparación: juzgado por LLM vs determinista

Si el run real completó muestras, esta celda compara los valores del juez con los deterministas (misma pregunta, misma variante) y resume acuerdos y desacuerdos. Si no completó ninguna, lo dice y las deterministas quedan como las cifras citables.

In [ ]:
import json

OUT = REPO + "/rag_baseline/ragas/metrics_ragas_llm_colab.json"
try:
    data = json.load(open(OUT))["results"]
except FileNotFoundError:
    print("No hay run real (no se corrio la celda 3 o no completo nada).")
    print("Las cifras citables son las deterministas de metrics_ragas.json.")
    data = {}

ok = {k: v for k, v in data.items() if v.get("status") == "ok"}
print(f"muestras juzgadas por LLM: {len(ok)}/36")
for k, v in sorted(ok.items()):
    lj, dt = v["llm_judged"], v["deterministic"]
    diffs = " | ".join(
        f"{m}: LLM {lj[m] if lj[m] is not None else 'null'} vs det "
        f"{dt[m] if dt.get(m) is not None else 'null'}"
        for m in ["context_precision", "context_recall", "faithfulness", "answer_relevancy"])
    print(f"{k}: {diffs}")